# rlmflow coding-agent walkthrough

Build a recursive coding agent, run a task, save the trace, and embed the generated artifact inline. This mirrors `examples/coding-agent/agent.py` but as a notebook so you can poke at every step.

Sibling notebooks read offline against the deterministic fixture under `examples/data/notebook-coding-agent/` (regenerated by `examples/data/build_notebook_fixture.py`):

- [`node_basics.ipynb`](./node_basics.ipynb) — querying the trace: `graph.walk`, `graph.find`, `graph.where`, `session.load_graph`, …
- [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb) — visualizing the trace: `tree`, `gantt`, `code_log`, mermaid / dot / d2, `report_md`, the inline Gradio viewer, etc.

Running this notebook end-to-end requires `OPENAI_API_KEY` and a live LLM call. Skip ahead to the other notebooks if you just want to consume the saved trace.

## 1. Build the agent

Four pieces snap together:

- `Workspace` — durable, branchable storage for sessions, traces, and any files the agent writes.
- `Runtime` — where REPL code runs. `LocalRuntime` runs in-process; `DockerRuntime` runs each step in a container.
- Tools — plain Python functions registered on the runtime. `FILE_TOOLS` gives the agent `read_file`, `write_file`, `edit_file`, `ls`, `grep`, etc.
- `RLMFlow` — wires an LLM client (plus optional cheaper alternates) to the runtime + workspace, and exposes `start` / `step` / `fork`. Every call to `start` / `step` returns a fresh immutable `Graph`.

In [10]:
from pathlib import Path

from rlmflow.llm import OpenAIClient
from rlmflow.prompts.default import DEFAULT_BUILDER
from rlmflow.rlm import RLMConfig, RLMFlow
from rlmflow.runtime.local import LocalRuntime
from rlmflow.tools import FILE_TOOLS
from rlmflow.workspace import Workspace

def build_agent(
    workspace_dir: str | Path = "./notebook-coding-agent",
    max_depth: int = 3,
    max_iterations: int = 30,
    max_concurrency: int = 8,
) -> RLMFlow:
    """Construct a coding agent identical to examples/coding-agent/agent.py."""
    workspace = Workspace.create(Path(workspace_dir).resolve())
    runtime = LocalRuntime(workspace=workspace)
    runtime.register_tools(FILE_TOOLS)

    return RLMFlow(
        llm_client=OpenAIClient("gpt-5-mini"),
        runtime=runtime,
        workspace=workspace,
        config=RLMConfig(
            max_depth=max_depth,
            max_iterations=max_iterations,
            max_concurrency=max_concurrency,
        ),
    )

## 2. Run a task

`agent.start(query)` returns the initial `Graph` (just the root agent with its `QueryNode`). Every `agent.step(graph)` advances exactly one tick — one LLM call, one REPL execution, or a wait-resolution — and returns a freshly-loaded `Graph` snapshot. Use `graph.current()` to inspect the latest state, `graph.finished` to stop, `graph.result()` for the terminal answer.

`live_view()` opens a self-updating Rich tree; you own the loop and just hand it the latest `Graph` on each tick. The agent loop and the visualization are decoupled.

In [11]:
TASK = """Create a runnable boids simulation in plain HTML, CSS, and JavaScript. It should render fast-moving and rainbow-colored boids on a 2D canvas.

Requirements:
- The main runnable interface is `index.html`.
- Split the implementation into separate components and files.
- Verify that all files were created and wired together before returning done, with no syntax errors.
"""

agent = build_agent(max_depth=2, workspace_dir="./boids-sim-workspace")
graph = agent.start(TASK)
graphs = [graph]

In [12]:
print(agent.build_system_prompt(graph))

## Role

You are answering a query with associated context in an iterative Python REPL.

## REPL

- Reply with exactly one fenced ` ```repl ` code block. Tools are already in the namespace.
- Think step by step in code/comments, plan briefly, then execute immediately.
- Output is truncated, so keep large data in variables/buffers and print summaries.
- Use Python to inspect, compute, branch, delegate, aggregate, and verify.
- Use only functions/tools that are actually present in the REPL or listed under `Tools`.

## Strategy

Inspect -> decompose -> batch -> wait -> verify -> done.

- Use the REPL as the work surface: inspect, compute, delegate, aggregate.
- If work splits into independent units, the parent fans out by default: chunks, documents, files, paths, records, trials, checks, components, artifacts, subproblems.
- Multi-file or multi-component artifacts are independent units unless there is a hard sequential dependency.
- Parent pattern: define shared contract + unit scopes -> 

In [13]:
from rlmflow.utils.viz import live_view

with live_view() as view:
    view(graph)
    while not graph.finished:
        graph = agent.step(graph)
        graphs.append(graph)
        view(graph)

Output()

In [14]:
current = graph.current()
print(f"{len(graphs)} snapshots  \u00b7  final: {graph.root_agent_id} [{current.type}]")
print(f"query : {graph.query[:120]!r}...")
print(f"result: {graph.result()[:200]}")

8 snapshots  ·  final: root [done_output]
query : 'Create a runnable boids simulation in plain HTML, CSS, and JavaScript. It should render fast-moving and rainbow-colored '...
result: All files created and passed basic static checks.

index.html (781 bytes):
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width,initia


In [15]:
print(graph.tree())

● root (default) — Create a runnable boids simulation in plain HTML, CSS, and …
  - [ 0] query
  - [ 1] llm
  - [ 6] llm code=# Continue verification and repair if n…
  - [ 2] llm code=# Plan: # 1. Spawn one child per requir…
  - [ 3] exec
  - [ 4] errored (exec_exception)
  - [ 5] llm
  - [ 6] llm code=# We'll (re)create the boids project fi…
  - [ 7] exec
  - [ 8] supervising waiting_on=['root.create_index.html', 'root.create_css/styles.css', 'root.create_js/boid.js', 'root.create_js/boids.js', 'root.create_js/main.js']
    ● root.create_index.html (default) — Write a runnable index.html for a fast rainbow-colored boid…
      - [ 0] query
      - [ 1] llm
      - [ 2] llm code=# Plan: # 1. Write the provided HTML co…
      - [ 3] exec
      - [ 4] done -> OK: index.html
    ● root.create_css/styles.css (default) — Write css/styles.css for the boids page. Requirements: - Da…
      - [ 0] query
      - [ 1] llm
      - [ 2] llm code=# Plan: # 1. Write the provided CSS con…
      - [ 3]

In [16]:
# from rlmflow.utils.trace import save_trace

# trace_path = save_trace(graphs, agent.workspace.trace_dir, metadata={"task": TASK})
# print(f"trace -> {trace_path}  ({len(graphs)} snapshots)")

## 3. Preview the generated artifact

Serve the generated `index.html` over local HTTP and embed that URL in the notebook. This exercises the same browser module-loading rules as opening the artifact normally, so import/export mistakes are not hidden by `srcdoc` inlining.

In [17]:
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from IPython.display import IFrame
import socket
import threading

live_candidates = sorted(agent.workspace.root.glob("**/index.html"))
canonical_candidates = sorted(Path("../data/boids-sim-workspace").glob("**/index.html"))
candidates = live_candidates or canonical_candidates
if not candidates:
    raise FileNotFoundError(f"no index.html under {agent.workspace.root}")

html_path = candidates[-1].resolve()
site_root = html_path.parent

if "preview_server" in globals():
    preview_server.shutdown()

with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]

class QuietHandler(SimpleHTTPRequestHandler):
    def log_message(self, *args):
        pass

handler = partial(QuietHandler, directory=str(site_root))
preview_server = ThreadingHTTPServer(("127.0.0.1", port), handler)
threading.Thread(target=preview_server.serve_forever, daemon=True).start()

url = f"http://127.0.0.1:{port}/{html_path.name}"
print(f"serving {site_root} -> {url}")
IFrame(url, width="100%", height=600)

serving /Users/shyam/Code/rlmkit/examples/notebooks/boids-sim-workspace -> http://127.0.0.1:53736/index.html


## 4. Open the interactive viewer

`open_viewer(source)` boots a Gradio stepper (step slider + clickable graph + per-agent transcript) inline.

In [9]:
from rlmflow.utils.viewer import open_viewer

open_viewer("./boids-sim-workspace", inline=True, height=720, quiet=True)

## 4. Render frames of each step

In [ ]:
# rlmflow render ./boids-sim-workspace \
#   --format steps \
#   --out ./boids-sim-workspace/frames \
#   --width 1600 \
#   --height 1200 \
#   --scale 2 \
#   --marker-mult 3.5 \
#   --text-mult 2.2

In [ ]:
from rlmflow.utils.viewer import save_steps

save_steps(
    "./boids-sim-workspace",
    "./boids-sim-workspace/frames",
    width=1600,
    height=1200,
    scale=2,
    marker_mult=3.5,
    text_mult=2.2,
)

## Next

- [`node_basics.ipynb`](./node_basics.ipynb) — walk the saved trace with the `Graph` query API.
- [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb) — inline Plotly, Mermaid, DOT, Gantt, report exports.